In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 17.9 MB/s eta 0:00:00


In [ ]:
"""
Run N4 Bias Field Correction
=============================
Your 'Dataset_N4' folder currently contains only skull-stripped (_bet) files —
N4 has not actually been run yet. This script runs it and saves results to a
NEW folder so your original skull-stripped files stay untouched.

Install once in Colab:
    !pip install SimpleITK

Folder structure assumed (based on your diagnostic output):
    Dataset_N4/1-Flair_bet.nii.gz, 1-T1_bet.nii.gz, 1-T2_bet.nii.gz, ... (60 patients)
    Some files duplicated as "Copy of X" — these are skipped automatically.

Output:
    N4_Corrected/1-Flair_n4.nii.gz, 1-T1_n4.nii.gz, 1-T2_n4.nii.gz, ...
"""

import os
import glob
import re
import SimpleITK as sitk

SKULL_STRIPPED_DIR = "/content/drive/MyDrive/MS_Classification_CNN_60Patients/Dataset_N4"
OUTPUT_DIR = "/content/drive/MyDrive/MS_Classification_CNN_60Patients/N4_Corrected"

MODALITIES = ["Flair", "T1", "T2"]

os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_real_patient_files():
    """
    Returns list of (patient_id, modality, filepath), skipping 'Copy of' duplicates.
    Matches pattern: '{id}-{modality}_bet.nii.gz'
    """
    all_files = glob.glob(os.path.join(SKULL_STRIPPED_DIR, "*_bet.nii.gz"))
    real_files = [f for f in all_files if "Copy of" not in os.path.basename(f)]

    parsed = []
    pattern = re.compile(r"^(\d+)-(Flair|T1|T2)_bet\.nii\.gz$")
    for f in real_files:
        match = pattern.match(os.path.basename(f))
        if match:
            patient_id, modality = match.groups()
            parsed.append((patient_id, modality, f))
        else:
            print(f"  Skipping unrecognized filename: {os.path.basename(f)}")

    return parsed


def run_n4(input_path, output_path):
    """Apply N4 bias field correction using SimpleITK."""
    img = sitk.ReadImage(input_path, sitk.sitkFloat32)

    # Since the image is already skull-stripped, background is ~0.
    # Build a mask from non-zero voxels so N4 only corrects inside the brain.
    mask = sitk.OtsuThreshold(img, 0, 1, 200)

    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected = corrector.Execute(img, mask)

    sitk.WriteImage(corrected, output_path)


def main():
    files = get_real_patient_files()
    print(f"Found {len(files)} real (non-duplicate) files to process.\n")

    for patient_id, modality, input_path in sorted(files, key=lambda x: (int(x[0]), x[1])):
        output_filename = f"{patient_id}-{modality}_n4.nii.gz"
        output_path = os.path.join(OUTPUT_DIR, output_filename)

        if os.path.exists(output_path):
            print(f"  [skip] {output_filename} already exists")
            continue

        print(f"  [N4] Patient {patient_id}, {modality} ...")
        try:
            run_n4(input_path, output_path)
            print(f"       -> saved {output_filename}")
        except Exception as e:
            print(f"       -> FAILED: {e}")

    print("\nDone. N4-corrected files saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

Found 180 real (non-duplicate) files to process.

  [skip] 1-Flair_n4.nii.gz already exists
  [skip] 1-T1_n4.nii.gz already exists
  [skip] 1-T2_n4.nii.gz already exists
  [skip] 2-Flair_n4.nii.gz already exists
  [skip] 2-T1_n4.nii.gz already exists
  [skip] 2-T2_n4.nii.gz already exists
  [skip] 3-Flair_n4.nii.gz already exists
  [skip] 3-T1_n4.nii.gz already exists
  [skip] 3-T2_n4.nii.gz already exists
  [skip] 4-Flair_n4.nii.gz already exists
  [skip] 4-T1_n4.nii.gz already exists
  [skip] 4-T2_n4.nii.gz already exists
  [skip] 5-Flair_n4.nii.gz already exists
  [skip] 5-T1_n4.nii.gz already exists
  [skip] 5-T2_n4.nii.gz already exists
  [skip] 6-Flair_n4.nii.gz already exists
  [skip] 6-T1_n4.nii.gz already exists
  [skip] 6-T2_n4.nii.gz already exists
  [skip] 7-Flair_n4.nii.gz already exists
  [skip] 7-T1_n4.nii.gz already exists
  [skip] 7-T2_n4.nii.gz already exists
  [skip] 8-Flair_n4.nii.gz already exists
  [skip] 8-T1_n4.nii.gz already exists
  [skip] 8-T2_n4.nii.gz alrea

KeyboardInterrupt: 